In [2]:
from datasets import load_dataset

ds = load_dataset("GEM/web_nlg", "en")

/Users/khangtuan/Documents/AdaKGC/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/khangtuan/Documents/AdaKGC/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
data = ds["test"].to_list()

In [16]:
for datum in data:
    if 'ALCO RS-3 has a diesel-electric transmission' in datum["target"]:
        print(datum)

{'gem_id': 'web_nlg_en-test-924', 'gem_parent_id': 'web_nlg_en-test-924', 'input': ['ALCO_RS-3 | buildDate | "May 1950 - August 1956"', 'ALCO_RS-3 | powerType | Diesel-electric_transmission', 'ALCO_RS-3 | builder | American_Locomotive_Company', 'ALCO_RS-3 | length | 17068.8 (millimetres)'], 'target': 'The American Locomotive Company built the ALCO RS-3 which was produced between May 1950 and August 1956. The ALCO RS-3 has a diesel-electric transmission and is 17068.8 millimetres in length.', 'references': ['The American Locomotive Company built the ALCO RS-3 which was produced between May 1950 and August 1956. The ALCO RS-3 has a diesel-electric transmission and is 17068.8 millimetres in length.', 'The length of the ALCO RS-3 is 17068.8 millimetres. It has a diesel-electric transmission. It was built and produced between May 1950 and August 1956 by the American Locomotive company.', 'The ALCO RS-3 was built by the American Locomotive Company and produced between May 1950 and August 195

In [13]:
import random
import ast
import csv

def load_dataset(file_path, num_fetch=None):

    all_records = []
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            for i, line in enumerate(file):
                if line.strip(): 
                    all_records.append((i + 1, line.strip()))
        
        if num_fetch is None or num_fetch >= len(all_records):
            print(f"Đã tải tất cả {len(all_records)} bản ghi")
            return all_records
        else:
            random_records = random.sample(all_records, num_fetch)
            print(f"Đã tải {num_fetch} bản ghi ngẫu nhiên từ tổng số {len(all_records)}")
            return random_records
            
    except Exception as e:
        print(f"Lỗi khi đọc tệp: {e}")
        return []

def load_triplets(file_path):

    triplets_by_line = []
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            for line in file:
                if line.strip():
                    triplets = ast.literal_eval(line.strip())
                    triplets_by_line.append(triplets)
        return triplets_by_line
    except Exception as e:
        print(f"Lỗi khi đọc tệp triplets: {e}")
        return []

def load_schema(file_path):

    schema_dict = {}
    try:
        with open(file_path, 'r', encoding='utf-8') as csvfile:
            reader = csv.reader(csvfile)
            for row in reader:
                if len(row) >= 2:
                    relation_name = row[0]
                    relation_definition = row[1]
                    schema_dict[relation_name] = relation_definition
        return schema_dict
    except Exception as e:
        print(f"Lỗi khi đọc file schema: {e}")
        return {}

def map_dataset_with_triplets_and_schema(dataset_tuples, triplets_file, schema_file):

    all_triplets = load_triplets(triplets_file)
    
    schema_dict = load_schema(schema_file)
    
    if not all_triplets:
        print("Không thể tải triplets")
        return dataset_tuples
    
    result = []
    for index, text in dataset_tuples:
        line_index = index - 1
        
        if 0 <= line_index < len(all_triplets):
            triplets = all_triplets[line_index]
            
            schema_mappings = []
            for triple in triplets:
                if len(triple) >= 2:
                    relation = triple[1]
                    definition = schema_dict.get(relation, "Không có định nghĩa")
                    schema_mappings.append((relation, definition))
            
            result.append((index, text, triplets, schema_mappings))
        else:
            print(f"Cảnh báo: Không tìm thấy triplets cho dòng {index}")
            result.append((index, text, [], []))
    
    return result



In [18]:
import json
def export_to_csv_and_text(mapped_data, csv_file, text_file):

    try:
        with open(csv_file, 'w', newline='', encoding='utf-8') as csvfile:
            fieldnames = ['index', 'text', 'triplets', 'schema_mappings']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            
            writer.writeheader()
            for index, text, triplets, schema_mappings in mapped_data:
                writer.writerow({
                    'index': index,
                    'text': text,
                    'triplets': json.dumps(triplets),  # Chuyển đổi list thành chuỗi JSON
                    'schema_mappings': json.dumps(schema_mappings)  # Chuyển đổi list thành chuỗi JSON
                })
        
        print(f"Đã xuất dữ liệu thành công vào file CSV: {csv_file}")
        
        with open(text_file, 'w', encoding='utf-8') as txtfile:
            for _, text, _, _ in mapped_data:
                txtfile.write(text + '\n')
        
        print(f"Đã xuất văn bản thành công vào file: {text_file}")
    
    except Exception as e:
        print(f"Lỗi khi xuất dữ liệu: {e}")


def process_and_export_dataset(dataset_file, triplets_file, schema_file, num_fetch=None, csv_output='dataset_1.csv', text_output='dataset_1_test.txt'):
    dataset_tuples = load_dataset(dataset_file, num_fetch)
    mapped_data = map_dataset_with_triplets_and_schema(dataset_tuples, triplets_file, schema_file)
    export_to_csv_and_text(mapped_data, csv_output, text_output)
    
    return mapped_data

In [19]:
process_and_export_dataset(
    'datasets/webnlg.txt', 
    'evaluate/references/webnlg.txt', 
    'schemas/webnlg_schema.csv',
    num_fetch=200,
    csv_output='dataset_1.csv',
    text_output='dataset_1_test.txt'
)

Đã tải 200 bản ghi ngẫu nhiên từ tổng số 1165
Đã xuất dữ liệu thành công vào file CSV: dataset_1.csv
Đã xuất văn bản thành công vào file: dataset_1_test.txt


[(1149,
  'Akeem Ayers was number 39 in the draft pick when he debuted for the Tennessee Titans.',
  [['Akeem_Ayers', 'draftPick', '"39"'],
   ['Akeem_Ayers', 'debutTeam', 'Tennessee_Titans']],
  [('draftPick',
    'The subject entity was selected as the number specified by the object entity in a draft.'),
   ('debutTeam',
    'The subject entity started their professional career with the object entity as their first team.')]),
 (1015,
  'First aired in October of 1983, the show Bananaman starred Bill Oddie. It was broadcast by the BBC, which was founded by John Reith and has its headquarters at Broadcasting House.',
  [['BBC', 'city', 'Broadcasting_House'],
   ['Bananaman', 'starring', 'Bill_Oddie'],
   ['BBC', 'foundedBy', 'John_Reith,_1st_Baron_Reith'],
   ['Bananaman', 'broadcastedBy', 'BBC'],
   ['Bananaman', 'firstAired', '"1983-10-03"']],
  [('city',
    'The subject entity is located in the city specified by the object entity.'),
   ('starring',
    'The subject entity features

In [20]:
import pandas as pd 
df = pd.read_csv('dataset_1.csv')
df.head()

,index,text,triplets,schema_mappings
0,1149,Akeem Ayers was number 39 in the draft pick wh...,"[[""Akeem_Ayers"", ""draftPick"", ""\""39\""""], [""Ake...","[[""draftPick"", ""The subject entity was selecte..."
1,1015,"First aired in October of 1983, the show Banan...","[[""BBC"", ""city"", ""Broadcasting_House""], [""Bana...","[[""city"", ""The subject entity is located in th..."
2,920,Ciudad Ayala is in the country of Mexico.,"[[""Ciudad_Ayala"", ""country"", ""Mexico""]]","[[""country"", ""The subject entity is located in..."
3,56,The Acharya Institute of Technology campus is ...,"[[""Acharya_Institute_of_Technology"", ""campus"",...","[[""campus"", ""The subject entity is located at ..."
4,745,"Established in the year 2000, the 11th Mississ...","[[""11th_Mississippi_Infantry_Monument"", ""estab...","[[""established"", ""The subject entity was estab..."


In [21]:
import csv
import json
import pandas as pd

def merge_csv_with_json(csv_file, json_file, output_csv):
    try:
        df_csv = pd.read_csv(csv_file)
        
        with open(json_file, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
        
        json_mapping = {}
        for item in json_data:
            input_text = item.get('input_text', '').strip()
            if input_text:
                json_mapping[input_text] = {
                    'schema_definition': item.get('schema_definition', {}),
                    'schema_canonicalization': item.get('schema_canonicalizaiton', []),
                    'oie': item.get('oie', [])
                }
        
        df_csv['schema_definition_json'] = df_csv['text'].apply(
            lambda x: json.dumps(json_mapping.get(x.strip(), {}).get('schema_definition', {}))
        )
        
        df_csv['schema_canonicalization'] = df_csv['text'].apply(
            lambda x: json.dumps(json_mapping.get(x.strip(), {}).get('schema_canonicalization', []))
        )
        
        df_csv['oie_json'] = df_csv['text'].apply(
            lambda x: json.dumps(json_mapping.get(x.strip(), {}).get('oie', []))
        )
        
        df_csv.to_csv(output_csv, index=False)
        
        print(f"Đã kết hợp thành công dữ liệu và lưu vào {output_csv}")
        
        matched_count = sum(1 for text in df_csv['text'] if text.strip() in json_mapping)
        print(f"Đã kết hợp {matched_count}/{len(df_csv)} bản ghi từ CSV với dữ liệu JSON")
        
    except Exception as e:
        print(f"Lỗi khi kết hợp dữ liệu: {e}")

def check_matching_rows(csv_file, json_file):
    try:
        df_csv = pd.read_csv(csv_file)
        csv_texts = set(text.strip() for text in df_csv['text'])
        
        with open(json_file, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
        
        json_texts = set(item.get('input_text', '').strip() for item in json_data)
        
        matched_texts = csv_texts.intersection(json_texts)
        
        print(f"Tổng số bản ghi trong CSV: {len(csv_texts)}")
        print(f"Tổng số bản ghi trong JSON: {len(json_texts)}")
        print(f"Số bản ghi khớp: {len(matched_texts)}")
        
        csv_only = csv_texts - json_texts
        json_only = json_texts - csv_texts
        
        if csv_only:
            print(f"\nVí dụ {min(5, len(csv_only))} bản ghi chỉ có trong CSV:")
            for i, text in enumerate(list(csv_only)[:5]):
                print(f"{i+1}. {text[:100]}...")
        
        if json_only:
            print(f"\nVí dụ {min(5, len(json_only))} bản ghi chỉ có trong JSON:")
            for i, text in enumerate(list(json_only)[:5]):
                print(f"{i+1}. {text[:100]}...")
                
    except Exception as e:
        print(f"Lỗi khi kiểm tra dữ liệu: {e}")

def process_and_merge_data(csv_file, json_file, output_csv):

    check_matching_rows(csv_file, json_file)
    
    merge_csv_with_json(csv_file, json_file, output_csv)


In [22]:
process_and_merge_data(
    './dataset_1.csv',
    './output/dataset_1_target_alignment/iter0/result_at_each_stage.json',
    './dataset_1_enriched.csv'
)

Tổng số bản ghi trong CSV: 200
Tổng số bản ghi trong JSON: 200
Số bản ghi khớp: 200
Đã kết hợp thành công dữ liệu và lưu vào ./dataset_1_enriched.csv
Đã kết hợp 200/200 bản ghi từ CSV với dữ liệu JSON


In [23]:
import pandas as pd 
df = pd.read_csv('dataset_1_enriched.csv')
df.head()

,index,text,triplets,schema_mappings,schema_definition_json,schema_canonicalization,oie_json
0,1149,Akeem Ayers was number 39 in the draft pick wh...,"[[""Akeem_Ayers"", ""draftPick"", ""\""39\""""], [""Ake...","[[""draftPick"", ""The subject entity was selecte...","{""draftPickNumber"": ""The subject entity was th...","[null, [""Akeem_Ayers"", ""affiliation"", ""Tenness...","[[""Akeem_Ayers"", ""draftPickNumber"", ""39""], [""A..."
1,1015,"First aired in October of 1983, the show Banan...","[[""BBC"", ""city"", ""Broadcasting_House""], [""Bana...","[[""city"", ""The subject entity is located in th...","{""firstAired"": ""The subject entity was first b...","[null, [""Bananaman"", ""firstAired"", ""October_19...","[[""Bananaman"", ""starring"", ""Bill_Oddie""], [""Ba..."
2,920,Ciudad Ayala is in the country of Mexico.,"[[""Ciudad_Ayala"", ""country"", ""Mexico""]]","[[""country"", ""The subject entity is located in...","{""country"": ""The subject entity is located in ...","[[""Ciudad_Ayala"", ""country"", ""Mexico""]]","[[""Ciudad_Ayala"", ""country"", ""Mexico""]]"
3,56,The Acharya Institute of Technology campus is ...,"[[""Acharya_Institute_of_Technology"", ""campus"",...","[[""campus"", ""The subject entity is located at ...","{""location"": ""The subject entity is located in...","[[""Acharya_Institute_of_Technology"", ""location...","[[""Acharya_Institute_of_Technology"", ""location..."
4,745,"Established in the year 2000, the 11th Mississ...","[[""11th_Mississippi_Infantry_Monument"", ""estab...","[[""established"", ""The subject entity was estab...","{""established"": ""The subject entity was establ...","[[""11th_Mississippi_Infantry_Monument"", ""estab...","[[""11th_Mississippi_Infantry_Monument"", ""estab..."


In [30]:
df.iloc[0].to_dict()

{'index': 1149,
 'text': 'Akeem Ayers was number 39 in the draft pick when he debuted for the Tennessee Titans.',
 'triplets': '[["Akeem_Ayers", "draftPick", "\\"39\\""], ["Akeem_Ayers", "debutTeam", "Tennessee_Titans"]]',
 'schema_mappings': '[["draftPick", "The subject entity was selected as the number specified by the object entity in a draft."], ["debutTeam", "The subject entity started their professional career with the object entity as their first team."]]',
 'schema_definition_json': '{"draftPickNumber": "The subject entity was the draft pick number specified by the object entity.", "team": "The subject entity is a member of the team specified by the object entity."}',
 'schema_canonicalization': '[null, ["Akeem_Ayers", "affiliation", "Tennessee_Titans"]]',
 'oie_json': '[["Akeem_Ayers", "draftPickNumber", "39"], ["Akeem_Ayers", "team", "Tennessee_Titans"]]'}

In [53]:
import pandas as pd
import json
import ast
import requests
import numpy as np
import ctranslate2
from transformers import AutoTokenizer
import torch
df = pd.read_csv('dataset_1_enriched.csv')

def calculate_distance(a, b):
    a = np.array(a)
    b = np.array(b)
    
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    
    similarity = dot_product / (norm_a * norm_b)
    return similarity


def get_embedding_sts(text: str,model="BGE", prompt_name=None, prompt=None, device="cpu"): 
    model_name = "BAAI/bge-base-en-v1.5"
    model_save_path = "bge_model_ctranslate2"
    # model_path = "bge_model_ctranslate2_base"


    device = "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if device == "cuda":
        translator = ctranslate2.Encoder(
            model_save_path, device=device, compute_type="float16"
        )  # or "cuda" for GPU
    else:
        translator = ctranslate2.Encoder(model_save_path, device=device)
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    input_ids = inputs["input_ids"].tolist()[0]
    tokens = tokenizer.convert_ids_to_tokens(input_ids)

    output = translator.forward_batch([tokens])

    
    last_hidden_state = output.last_hidden_state
    last_hidden_state = np.array(last_hidden_state)
    last_hidden_state = torch.as_tensor(last_hidden_state, device = device)[0]

    last_hidden_state = torch.nn.functional.normalize(last_hidden_state, p=2, dim=1)

    if device == "cuda":
        embeddings = last_hidden_state.detach().cpu().tolist()[0]
    else:
        embeddings = last_hidden_state.detach().tolist()[0]

    return embeddings


def determine_nota(row):
    notas = []
    try:
        sc = row['schema_canonicalization']
        oie = row['oie_json']
        sm = json.loads(row["schema_mappings"])
        sd = json.loads(row["schema_definition_json"])
        sc_json = (json.loads(sc))
        oie_json = json.loads(oie)
        notas = []
        for i in range(len(sc_json)):
            if sc_json[i] is None or sc_json[i] == "null":
                notas.append(True)
            elif calculate_distance(get_embedding_sts(sd[sc_json[i][1]]), get_embedding_sts(sm[oie_json[i][1]])) > 0.5:
                notas.append(False)
            else:
                notas.append(True)
        return notas
    except Exception as e:
        print(f"Error processing row: {e}")
        return notas

df['NOTA'] = df.apply(determine_nota, axis=1)

df.to_csv('dataset_1_nota.csv', index=False)
# print(calculate_distance(get_embedding_sts("positive"), get_embedding_sts("positive")))

Error processing row: 'affiliation'
Error processing row: list indices must be integers or slices, not str


KeyboardInterrupt: 

In [38]:
import pandas as pd 
df = pd.read_csv('dataset_1_nota.csv')
df.head()

,index,text,triplets,schema_mappings,schema_definition_json,schema_canonicalization,oie_json,NOTA
0,1149,Akeem Ayers was number 39 in the draft pick wh...,"[[""Akeem_Ayers"", ""draftPick"", ""\""39\""""], [""Ake...","[[""draftPick"", ""The subject entity was selecte...","{""draftPickNumber"": ""The subject entity was th...","[null, [""Akeem_Ayers"", ""affiliation"", ""Tenness...","[[""Akeem_Ayers"", ""draftPickNumber"", ""39""], [""A...","[True, True]"
1,1015,"First aired in October of 1983, the show Banan...","[[""BBC"", ""city"", ""Broadcasting_House""], [""Bana...","[[""city"", ""The subject entity is located in th...","{""firstAired"": ""The subject entity was first b...","[null, [""Bananaman"", ""firstAired"", ""October_19...","[[""Bananaman"", ""starring"", ""Bill_Oddie""], [""Ba...","[True, False, True, True, True]"
2,920,Ciudad Ayala is in the country of Mexico.,"[[""Ciudad_Ayala"", ""country"", ""Mexico""]]","[[""country"", ""The subject entity is located in...","{""country"": ""The subject entity is located in ...","[[""Ciudad_Ayala"", ""country"", ""Mexico""]]","[[""Ciudad_Ayala"", ""country"", ""Mexico""]]",[False]
3,56,The Acharya Institute of Technology campus is ...,"[[""Acharya_Institute_of_Technology"", ""campus"",...","[[""campus"", ""The subject entity is located at ...","{""location"": ""The subject entity is located in...","[[""Acharya_Institute_of_Technology"", ""location...","[[""Acharya_Institute_of_Technology"", ""location...","[False, False, False]"
4,745,"Established in the year 2000, the 11th Mississ...","[[""11th_Mississippi_Infantry_Monument"", ""estab...","[[""established"", ""The subject entity was estab...","{""established"": ""The subject entity was establ...","[[""11th_Mississippi_Infantry_Monument"", ""estab...","[[""11th_Mississippi_Infantry_Monument"", ""estab...","[False, False, False, True]"


In [1]:
import pandas as pd 
df = pd.read_csv('complete_dataset.csv')
df.head()

,index,text,canon_kg,labeled_schema,labeled_schema_definition,reference,oie,schema_definition,schema_canonicalization,nota
0,1,"The location of Trane is Swords, Dublin.","[[""Trane"", ""location"", ""Swords,_Dublin""]]","[""location""]","[""The subject entity is located in the place s...",NaN,"[[""Trane"", ""location"", ""Swords,_Dublin""]]","{""location"": ""Text: The first store opened in ...","[[""Trane"", ""location"", ""Swords,_Dublin""]]",False
1,2,"The Ciudad Ayala city, a part of Morelos with ...","[[""Ciudad_Ayala"", ""populationMetro"", ""1777539""...","[""populationMetro"", ""type"", ""governmentType""]","[""The subject entity has a metropolitan popula...",NaN,"[[""Ciudad_Ayala"", ""populationMetro"", ""1777539""...","{""populationDensity"": ""Text: It has a populati...","[[""Ciudad_Ayala"", ""populationMetro"", ""1777539""...",False
2,3,The 17068.8 millimeter long ALCO RS-3 has a di...,[],[],[],NaN,"[[""ALCO_RS-3"", ""length"", ""17068.8 (millimetres...","{""length"": ""Text: The length of the main span ...","[null, null]",True
3,4,"Alan B. Miller Hall, in Virginia, USA, was des...","[[""Alan_B._Miller_Hall"", ""architect"", ""Robert_...","[""architect"", ""address"", ""currentTenants"", ""lo...","[""The subject entity was designed or planned b...",NaN,"[[""Alan_B._Miller_Hall"", ""architect"", ""Robert_...","{""country"": ""Text: The first season of the Ame...","[[""Alan_B._Miller_Hall"", ""architect"", ""Robert_...",False
4,5,Liselotte Grschebina was born in Karlsruhe and...,"[[""Liselotte_Grschebina"", ""birthPlace"", ""Karls...","[""birthPlace""]","[""The subject entity was born in the location ...",NaN,"[[""Liselotte_Grschebina"", ""bornIn"", ""Karlsruhe...","{""diedIn"": ""Text: John Le Mesurier (born John ...","[[""Liselotte_Grschebina"", ""birthPlace"", ""Karls...",False


In [8]:
test_record = df.iloc[16].to_dict()

In [9]:
test_record


{'index': 17,
 'text': 'The Estadio Municipal Coaracy da Mata Fonseca in Arapiraca is the ground of Agremiação Sportiva Arapiraquense (which has 17000 members) that play in the Campeonato Brasileiro Série C league. Champions, The Vila Nova Futebol Clube at Campeonato Brasileiro Série C from Brazil.',
 'canon_kg': '[["Estadio_Municipal_Coaracy_da_Mata_Fonseca", "location", "Arapiraca"], ["Agremia\\u00e7\\u00e3o_Sportiva_Arapiraquense", "numberOfMembers", "17000"], ["Agremia\\u00e7\\u00e3o_Sportiva_Arapiraquense", "league", "Campeonato_Brasileiro_S\\u00e9rie_C"], ["Vila_Nova_Futebol_Clube", "league", "Campeonato_Brasileiro_S\\u00e9rie_C"], ["Campeonato_Brasileiro_S\\u00e9rie_C", "country", "Brazil"]]',
 'labeled_schema': '["location", "numberOfMembers", "league", "league", "country"]',
 'labeled_schema_definition': '["The subject entity is located in the place specified by the object entity.", "The subject entity has the number of members as specified by the object entity.", "The subject

In [ ]:
# so cai cannon_kg voi schema_canonicalization

test_record["canon_kg"]

test_record["schema_canonicalization"]